# 02. FT-Transformer v2 Reproduction

Kaggle **Predicting Student Health Risk (Playground Series S6E7)**에서 사용한 FT-Transformer v2 전체 재현 코드입니다.

이 노트북은 단순한 모델 학습 코드가 아니라 다음 흐름을 모두 포함합니다.

**원본 13개 feature → exact-value multiclass Target Encoding 39개 추가 → 총 52개 input → 7-fold FT-Transformer 학습 → OOF/test probability 생성 → 여러 decision rule 비교 → 최종 submission 생성**

최종 제출에서는 **fixed inverse-prior correction**이 가장 높은 점수를 기록했습니다.

- Private Balanced Accuracy: **0.95066**
- Public Balanced Accuracy: **0.95054**


## 1. 실행 설정 · 경로 · 공통 유틸리티

먼저 전체 reproduction에 필요한 실행 옵션과 공통 함수를 정의합니다.

- **7-fold Stratified CV**: 클래스 비율을 유지하면서 7개 fold로 나눕니다.
- **4-member ensemble**: 각 fold에서 FT-Transformer를 4번 학습해 probability를 평균합니다.
- **16 epochs**: 모든 fold에서 동일한 epoch 수를 사용합니다.
- `--resume`: 이미 끝난 fold의 cache가 있으면 다시 학습하지 않고 재사용합니다.
- `fast_balanced_accuracy`, `recalls`: 이 대회의 핵심 평가지표와 클래스별 recall을 계산합니다.
- `normalize_probability`: 각 행의 class probability 합이 1이 되도록 정규화합니다.

또한 데이터 경로와 출력 경로를 자동으로 찾도록 해 Kaggle/로컬 환경 모두에서 재현할 수 있게 했습니다.


In [ ]:
# Full reproduction settings. For a smoke test, change n_ens/epochs/folds.
import sys
sys.argv = [
    "05_ft_transformer_v2_reproduction.py",
    "--data-dir", "/kaggle/input/competitions/playground-series-s6e7",
    "--output-dir", "/kaggle/working/ftt_v2_outputs",
    "--n-splits", "7",
    "--n-ens", "4",
    "--n-epochs", "16",
    "--device", "cuda",
    "--resume",
]

import argparse
import gc
import hashlib
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

TARGET = "health_condition"
ID_COL = "id"
OUTER_SEED = 42
TE_SEED = 42
MODEL_SEED = 42
DEFAULT_N_SPLITS = 7
DEFAULT_N_ENS = 4
DEFAULT_N_EPOCHS = 16
TE_CV = 5

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="S6E7 FT-Transformer-v2 reproduction with fold-local exact-value TE")
    parser.add_argument("--data-dir", type=Path, default=None)
    parser.add_argument("--output-dir", type=Path, default=None)
    parser.add_argument("--n-splits", type=int, default=DEFAULT_N_SPLITS)
    parser.add_argument("--n-ens", type=int, default=DEFAULT_N_ENS)
    parser.add_argument("--n-epochs", type=int, default=DEFAULT_N_EPOCHS)
    parser.add_argument("--resume", action="store_true", help="Reuse completed fold caches from output-dir/folds.")
    parser.add_argument("--device", default="cuda", help="masamlp device. The faithful full run expects cuda.")
    return parser.parse_args()

def resolve_data_dir(requested: Path | None) -> Path:
    candidates = []
    if requested is not None:
        candidates.append(requested)
    env_path = os.environ.get("S6E7_DATA_DIR")
    if env_path:
        candidates.append(Path(env_path))
    script_root = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
    candidates.extend([Path("/kaggle/input/playground-series-s6e7"), script_root / "downloads" / "DScover 가이드 프로젝트", Path.cwd() / "downloads" / "DScover 가이드 프로젝트"])
    for candidate in candidates:
        if all((candidate / name).exists() for name in ["train.csv", "test.csv"]):
            return candidate.resolve()
    raise FileNotFoundError("Could not locate train.csv/test.csv. Pass --data-dir or set S6E7_DATA_DIR.")

def resolve_output_dir(requested: Path | None) -> Path:
    if requested is not None:
        return requested.resolve()
    if Path("/kaggle/working").exists():
        return Path("/kaggle/working/ftt_v2_outputs")
    return Path(__file__).resolve().parent / "ftt_v2_outputs"

def array_sha256(values: np.ndarray) -> str:
    contiguous = np.ascontiguousarray(values)
    return hashlib.sha256(contiguous.view(np.uint8)).hexdigest()

def fast_balanced_accuracy(y_true: np.ndarray, prediction: np.ndarray) -> float:
    counts = np.bincount(y_true, minlength=3)
    correct = np.bincount(y_true[y_true == prediction], minlength=3)
    return float(np.mean(correct / counts))

def recalls(y_true: np.ndarray, prediction: np.ndarray) -> list[float]:
    return [float(np.mean(prediction[y_true == c] == c)) for c in range(3)]

def normalize_probability(proba: np.ndarray) -> np.ndarray:
    proba = np.asarray(proba, dtype=np.float64)
    proba = np.clip(proba, 1e-12, None)
    return proba / proba.sum(axis=1, keepdims=True)


## 2. Exact-value Multiclass Target Encoding

원본 feature 13개 각각에 대해 **3개 클래스와의 관계를 나타내는 Target Encoding**을 만듭니다.

- 13개 feature × 3 classes = **39개 TE feature**
- 원본 13개 + TE 39개 = **총 52개 model input**

여기서 중요한 점은 **target leakage 방지**입니다.  
Outer validation fold의 정답을 encoding에 사용하지 않도록, 각 outer training fold 내부에서 다시 **5-fold cross-fitting**으로 Target Encoding을 생성합니다.

쉽게 말하면, 모델에게 단순히 `stress_level = high`라는 값만 주는 것이 아니라  
“이 값이 학습 데이터에서 각 건강 상태와 얼마나 연결되어 있었는가?”라는 통계적 힌트도 함께 주는 방식입니다.


In [ ]:
def add_exact_value_te(
    fit_raw: pd.DataFrame,
    valid_raw: pd.DataFrame,
    test_raw: pd.DataFrame,
    y_fit: np.ndarray,
    feature_columns: list[str],
    fold: int,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    from catstat import TargetEncoder
    encoder = TargetEncoder(cols=feature_columns, stats=("mean",), target_type="multiclass", smooth="auto", numeric="direct", cv=TE_CV, random_state=TE_SEED, output="numpy")
    fit_te = np.asarray(encoder.fit_transform(fit_raw[feature_columns], y_fit), dtype=np.float32)
    valid_te = np.asarray(encoder.transform(valid_raw[feature_columns]), dtype=np.float32)
    test_te = np.asarray(encoder.transform(test_raw[feature_columns]), dtype=np.float32)
    expected = len(feature_columns) * 3
    if fit_te.shape[1] != expected:
        raise ValueError(f"Expected {expected} TE columns, got {fit_te.shape[1]}")
    names = [f"exact_te_{j:02d}" for j in range(expected)]
    def attach(raw: pd.DataFrame, encoded: np.ndarray) -> pd.DataFrame:
        raw = raw[feature_columns].reset_index(drop=True).copy()
        te = pd.DataFrame(encoded, columns=names, index=raw.index)
        return pd.concat([raw, te], axis=1)
    return attach(fit_raw, fit_te), attach(valid_raw, valid_te), attach(test_raw, test_te)


## 3. FT-Transformer 모델 정의

이 부분에서 실제 FT-Transformer 구조와 학습 설정을 지정합니다.

주요 설정:

- `d_block = 128`
- Transformer block 2개
- Attention head 8개
- batch size 4096
- learning rate 0.001
- AdamW optimizer + cosine scheduler
- 숫자형 변수: quantile scaling + PLR-lite embedding
- 범주형 변수: embedding
- fold당 `n_ens = 4`

FT-Transformer를 선택한 이유는 **LightGBM과 다른 방식으로 변수 간 interaction을 학습하기 위해서**입니다.  
수면, 스트레스, 운동, BMI처럼 여러 변수가 함께 작용할 가능성이 있는 문제에서 attention 기반 feature interaction을 비교해볼 수 있습니다.

`reorder_model_probability()`는 모델이 반환하는 class 순서를 고정된 `[0,1,2]` 순서로 맞춰 OOF/test probability가 뒤섞이지 않게 합니다.


In [ ]:
def make_model(categorical_columns: list[str], n_ens: int, n_epochs: int, device: str):
    from masamlp import MasaClassifier
    return MasaClassifier(model="ft_transformer", model_params={"d_block": 128, "n_blocks": 2, "attention_n_heads": 8}, n_epochs=n_epochs, batch_size=4096, eval_batch_size=8192, learning_rate=1e-3, weight_decay=1e-5, optimizer="adamw", lr_scheduler="cosine", num_embedding="plr-lite", numeric_scaler="quantile", categorical_features=categorical_columns, cat_encoding="embedding", n_ens=n_ens, ens_mode="loop", early_stopping_rounds=None, class_weight=None, device=device, amp="auto", verbose=1, random_state=MODEL_SEED)

def reorder_model_probability(model, proba: np.ndarray) -> np.ndarray:
    model_classes = np.asarray(model.classes_)
    desired = np.arange(3)
    positions = []
    for label in desired:
        match = np.flatnonzero(model_classes == label)
        if len(match) != 1:
            raise ValueError(f"Unexpected model classes: {model_classes.tolist()}")
        positions.append(int(match[0]))
    return normalize_probability(proba[:, positions]).astype(np.float32)


## 4. Class imbalance 보정용 Decision Rules

EDA에서 `at-risk`가 약 86%로 매우 많다는 것을 확인했습니다.  
하지만 평가지표는 **Balanced Accuracy**이므로 다수 클래스만 잘 맞히는 것보다 세 클래스의 recall을 균형 있게 높이는 것이 중요합니다.

여기서는 세 가지 방식의 decision rule을 준비합니다.

1. **Raw argmax**  
   모델 probability 중 가장 큰 클래스를 그대로 선택합니다.

2. **Inverse-prior / prior-beta**  
   `probability / prior^beta` 형태로 다수 클래스의 영향은 낮추고 minority class를 상대적으로 강화합니다.  
   `beta=1`이면 fixed inverse-prior입니다.

3. **Class multiplier**  
   클래스별 probability에 서로 다른 multiplier를 곱해 Balanced Accuracy가 높아지는 조합을 탐색합니다.

`differential_evolution`을 사용해 beta와 multiplier를 최적화합니다.


In [ ]:
def prior_prediction(proba: np.ndarray, prior: np.ndarray, beta: float = 1.0) -> np.ndarray:
    return np.argmax(proba / np.power(prior, beta), axis=1)

def tune_prior_beta(y_true: np.ndarray, proba: np.ndarray, prior: np.ndarray, seed: int) -> tuple[float, float]:
    def objective(vector: np.ndarray) -> float:
        return -fast_balanced_accuracy(y_true, prior_prediction(proba, prior, beta=float(vector[0])))
    result = differential_evolution(objective, bounds=[(0.5, 1.5)], seed=seed, popsize=10, maxiter=18, polish=False, updating="immediate", workers=1)
    beta = float(result.x[0])
    return beta, -float(result.fun)

def tune_multipliers(y_true: np.ndarray, proba: np.ndarray, prior: np.ndarray, seed: int) -> tuple[np.ndarray, float]:
    initial = np.log((1.0 / prior) / (1.0 / prior)[0])[1:]
    def objective(log_ratios: np.ndarray) -> float:
        multipliers = np.exp(np.r_[0.0, log_ratios])
        return -fast_balanced_accuracy(y_true, np.argmax(proba * multipliers, axis=1))
    result = differential_evolution(objective, bounds=[(float(initial[0] - 0.75), float(initial[0] + 0.75)), (float(initial[1] - 0.75), float(initial[1] + 0.75))], seed=seed, popsize=7, maxiter=18, polish=False, updating="immediate", workers=1)
    multipliers = np.exp(np.r_[0.0, result.x])
    return multipliers, -float(result.fun)


## 5. Post-processing의 Cross-fitting

Decision rule을 전체 OOF에 맞춰 최적화한 뒤 같은 OOF에서 점수를 재면 성능을 낙관적으로 볼 수 있습니다.

그래서 `crossfit_decisions()`에서는:

- 7개 OOF fold 중 6개 fold로 beta/multiplier를 튜닝하고
- 남은 1개 fold에서 실제 성능을 평가한 뒤
- 이를 7번 반복합니다.

즉, **post-processing 자체도 validation leakage 없이 평가**하려는 구조입니다.

이 때문에 full-OOF 최적화 점수는 뒤에서 명시적으로 `optimistic`이라고 구분합니다.


In [ ]:
def crossfit_decisions(y: np.ndarray, oof: np.ndarray, fold_id: np.ndarray, n_splits: int) -> tuple[np.ndarray, np.ndarray, list[dict]]:
    beta_prediction = np.empty(len(y), dtype=np.int8)
    multiplier_prediction = np.empty(len(y), dtype=np.int8)
    rows = []
    for heldout in range(n_splits):
        meta_fit = fold_id != heldout
        meta_valid = fold_id == heldout
        prior = np.bincount(y[meta_fit], minlength=3).astype(np.float64)
        prior /= prior.sum()
        beta, beta_train_score = tune_prior_beta(y[meta_fit], oof[meta_fit], prior, 7000 + heldout)
        multipliers, multiplier_train_score = tune_multipliers(y[meta_fit], oof[meta_fit], prior, 8000 + heldout)
        beta_prediction[meta_valid] = prior_prediction(oof[meta_valid], prior, beta)
        multiplier_prediction[meta_valid] = np.argmax(oof[meta_valid] * multipliers, axis=1)
        rows.append({"heldout_fold": heldout + 1, "prior": prior.tolist(), "beta": beta, "beta_meta_fit_score": beta_train_score, "beta_heldout_score": fast_balanced_accuracy(y[meta_valid], beta_prediction[meta_valid]), "multipliers": multipliers.tolist(), "multiplier_meta_fit_score": multiplier_train_score, "multiplier_heldout_score": fast_balanced_accuracy(y[meta_valid], multiplier_prediction[meta_valid])})
    return beta_prediction, multiplier_prediction, rows


## 6. 평가 · 제출 · Fold Cache 보조 함수

이 구간은 실험을 안정적으로 관리하기 위한 함수들입니다.

- `metric_row()`: Balanced Accuracy와 클래스별 recall을 한 행으로 정리
- `save_submission()`: Kaggle 제출 형식을 검증한 뒤 CSV 저장
- `fold_cache_matches()`: `--resume` 시 이전 fold 결과가 현재 데이터/설정과 정확히 같은지 검사

특히 cache 검증을 넣은 이유는 **다른 설정으로 만든 probability를 실수로 재사용하는 것을 막기 위해서**입니다.


In [ ]:
def metric_row(name: str, y: np.ndarray, prediction: np.ndarray) -> dict:
    return {"candidate": name, "balanced_accuracy": fast_balanced_accuracy(y, prediction), "at_risk_recall": recalls(y, prediction)[0], "fit_recall": recalls(y, prediction)[1], "unhealthy_recall": recalls(y, prediction)[2]}

def save_submission(sample: pd.DataFrame, label_encoder: LabelEncoder, prediction: np.ndarray, path: Path) -> None:
    submission = sample.copy()
    submission[TARGET] = label_encoder.inverse_transform(prediction)
    if list(submission.columns) != [ID_COL, TARGET]:
        raise ValueError(f"Unexpected submission columns: {submission.columns.tolist()}")
    if submission[ID_COL].duplicated().any() or submission[TARGET].isna().any():
        raise ValueError("Invalid submission: duplicate ID or missing prediction")
    submission.to_csv(path, index=False)

def fold_cache_matches(cached, valid_idx: np.ndarray, train_ids: np.ndarray, test_ids: np.ndarray, config_json: str) -> bool:
    return np.array_equal(cached["valid_idx"], valid_idx) and np.array_equal(cached["valid_ids"], train_ids[valid_idx]) and np.array_equal(cached["test_ids"], test_ids) and str(cached["config_json"].item()) == config_json


## 7. 데이터 로드 및 실험 설정 기록

`main()`의 첫 부분입니다.

여기서는:

- train / test / sample submission 로드
- feature와 categorical feature 구분
- target을 0,1,2로 Label Encoding
- class prior 계산 준비
- 실험 설정을 `ftt_v2_config.json`으로 저장

을 수행합니다.

모델 구조, seed, fold 수, epoch, optimizer, package version까지 기록해 두므로 나중에 **어떤 설정으로 만든 결과인지 재현 가능**합니다.


In [ ]:
def main() -> None:
    args = parse_args()
    if args.n_splits < 2 or args.n_ens < 1 or args.n_epochs < 1:
        raise ValueError("n-splits >= 2, n-ens >= 1, and n-epochs >= 1 are required")
    data_dir = resolve_data_dir(args.data_dir)
    output_dir = resolve_output_dir(args.output_dir)
    fold_dir = output_dir / "folds"
    output_dir.mkdir(parents=True, exist_ok=True)
    fold_dir.mkdir(parents=True, exist_ok=True)
    import catstat, masamlp, torch
    if args.device.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available. Use a Kaggle GPU session for the full run.")
    started = time.time()
    train = pd.read_csv(data_dir / "train.csv")
    test = pd.read_csv(data_dir / "test.csv")
    sample_path = data_dir / "sample_submission.csv"
    sample = pd.read_csv(sample_path) if sample_path.exists() else test[[ID_COL]].assign(**{TARGET: ""})
    feature_columns = [c for c in test.columns if c != ID_COL]
    categorical_columns = [c for c in feature_columns if not pd.api.types.is_numeric_dtype(train[c])]
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(train[TARGET]).astype(np.int8)
    classes = label_encoder.classes_.astype(str)
    train_ids = train[ID_COL].to_numpy()
    test_ids = test[ID_COL].to_numpy()
    raw_train = train[feature_columns]
    raw_test = test[feature_columns]
    config = {"data_rows": [len(train), len(test)], "feature_columns": feature_columns, "categorical_columns": categorical_columns, "outer_n_splits": args.n_splits, "outer_seed": OUTER_SEED, "te_cv": TE_CV, "te_seed": TE_SEED, "te_smooth": "auto", "te_numeric": "direct", "model": "ft_transformer", "model_params": {"d_block": 128, "n_blocks": 2, "attention_n_heads": 8}, "n_ens": args.n_ens, "n_epochs": args.n_epochs, "batch_size": 4096, "learning_rate": 0.001, "weight_decay": 0.00001, "optimizer": "adamw", "lr_scheduler": "cosine", "numeric_scaler": "quantile", "num_embedding": "plr-lite", "model_seed": MODEL_SEED, "device": args.device, "catstat_version": getattr(catstat, "__version__", "unknown"), "masamlp_version": getattr(masamlp, "__version__", "unknown"), "torch_version": torch.__version__}
    config_json = json.dumps(config, sort_keys=True)
    (output_dir / "ftt_v2_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")


## 8. 7-fold FT-Transformer 학습 및 OOF/Test Probability 생성

이 부분이 실제 모델 학습의 핵심입니다.

각 fold마다 다음 순서로 진행됩니다.

1. Outer train / validation 분리
2. 해당 fold의 training data만 이용해 Target Encoding 생성
3. 원본 13개 + TE 39개 = 52개 feature 구성
4. FT-Transformer 학습
5. validation probability → OOF 위치에 저장
6. test probability → fold별로 누적
7. fold별 Balanced Accuracy와 LogLoss 기록
8. fold 결과를 `.npz` cache로 저장

7개 fold가 끝나면 test probability를 평균해 최종 test probability를 만듭니다.

또한 validation fold를 epoch 선택에 사용하지 않고 **고정 16 epochs**로 학습해, validation 정보가 학습 과정에 개입하지 않도록 했습니다.


In [ ]:
    oof = np.zeros((len(train), 3), dtype=np.float32)
    test_sum = np.zeros((len(test), 3), dtype=np.float64)
    fold_id = np.full(len(train), -1, dtype=np.int8)
    fold_rows = []
    outer = StratifiedKFold(n_splits=args.n_splits, shuffle=True, random_state=OUTER_SEED)
    folds = list(outer.split(raw_train, y))
    for fold, (fit_idx, valid_idx) in enumerate(folds, start=1):
        fold_started = time.time()
        fold_id[valid_idx] = fold - 1
        fold_path = fold_dir / f"ftt_v2_fold_{fold:02d}.npz"
        if args.resume and fold_path.exists():
            cached = np.load(fold_path, allow_pickle=False)
            if not fold_cache_matches(cached, valid_idx, train_ids, test_ids, config_json):
                raise ValueError(f"Fold cache does not match current run: {fold_path}")
            valid_proba = cached["valid_proba"].astype(np.float32)
            fold_test_proba = cached["test_proba"].astype(np.float32)
            source = "cache"
        else:
            fit_raw = raw_train.iloc[fit_idx].reset_index(drop=True)
            valid_raw = raw_train.iloc[valid_idx].reset_index(drop=True)
            test_raw = raw_test.reset_index(drop=True)
            fit_frame, valid_frame, test_frame = add_exact_value_te(fit_raw, valid_raw, test_raw, y[fit_idx], feature_columns, fold)
            if fit_frame.shape[1] != 52:
                raise ValueError(f"Expected 52 model inputs, got {fit_frame.shape[1]}")
            model = make_model(categorical_columns, n_ens=args.n_ens, n_epochs=args.n_epochs, device=args.device)
            model.fit(fit_frame, y[fit_idx])
            valid_proba = reorder_model_probability(model, model.predict_proba(valid_frame))
            fold_test_proba = reorder_model_probability(model, model.predict_proba(test_frame))
            np.savez_compressed(fold_path, valid_idx=valid_idx, valid_ids=train_ids[valid_idx], test_ids=test_ids, valid_proba=valid_proba, test_proba=fold_test_proba, classes=classes, config_json=np.asarray(config_json))
            source = "trained"
            del model, fit_frame, valid_frame, test_frame, fit_raw, valid_raw, test_raw
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        oof[valid_idx] = valid_proba
        test_sum += fold_test_proba
        fold_prediction = valid_proba.argmax(axis=1)
        fold_row = {"fold": fold, "source": source, "raw_balanced_accuracy": fast_balanced_accuracy(y[valid_idx], fold_prediction), "raw_logloss": log_loss(y[valid_idx], valid_proba, labels=np.arange(3)), "seconds": time.time() - fold_started}
        fold_rows.append(fold_row)
        print(json.dumps(fold_row), flush=True)
        del valid_proba, fold_test_proba
        gc.collect()


## 9. OOF 기반 Decision Rule 비교 및 Submission 생성

모델 학습이 끝난 뒤 같은 probability에 여러 decision rule을 적용해 비교합니다.

비교 대상:

- `ftt_raw_argmax`
- `ftt_fixed_inverse_prior`
- `ftt_prior_beta_crossfit`
- `ftt_unrestricted_multiplier_crossfit`
- full-OOF beta / multiplier 결과 (`optimistic` 표시)

그리고 test set에 대해 네 가지 제출파일을 생성합니다.

- `submission_ftt_v2_raw.csv`
- `submission_ftt_v2_inverse_prior.csv`
- `submission_ftt_v2_prior_beta.csv`
- `submission_ftt_v2_multiplier.csv`

실제 Kaggle 제출에서는 **inverse-prior variant가 가장 높은 점수**를 기록했습니다.

추가로 OOF/test probability, fold metrics, post-processing 결과, config와 SHA256 hash까지 저장해 이후 ensemble이나 reproduction에 다시 사용할 수 있게 합니다.


In [ ]:
    if np.any(fold_id < 0) or np.any(oof.sum(axis=1) <= 0):
        raise RuntimeError("OOF matrix is incomplete")
    test_proba = normalize_probability(test_sum / args.n_splits).astype(np.float32)
    oof = normalize_probability(oof).astype(np.float32)
    prior = np.bincount(y, minlength=3).astype(np.float64) / len(y)
    raw_prediction = oof.argmax(axis=1)
    inverse_prior_prediction = prior_prediction(oof, prior, beta=1.0)
    beta_crossfit_prediction, multiplier_crossfit_prediction, post_rows = crossfit_decisions(y, oof, fold_id, args.n_splits)
    full_beta, full_beta_score = tune_prior_beta(y, oof, prior, seed=9001)
    full_multipliers, full_multiplier_score = tune_multipliers(y, oof, prior, seed=9002)
    full_beta_prediction = prior_prediction(oof, prior, full_beta)
    full_multiplier_prediction = np.argmax(oof * full_multipliers, axis=1)
    comparison = pd.DataFrame([metric_row("ftt_raw_argmax", y, raw_prediction), metric_row("ftt_fixed_inverse_prior", y, inverse_prior_prediction), metric_row("ftt_prior_beta_crossfit", y, beta_crossfit_prediction), metric_row("ftt_unrestricted_multiplier_crossfit", y, multiplier_crossfit_prediction), metric_row("ftt_prior_beta_full_oof_optimistic", y, full_beta_prediction), metric_row("ftt_multiplier_full_oof_optimistic", y, full_multiplier_prediction)])
    comparison.to_csv(output_dir / "ftt_v2_comparison.csv", index=False)
    pd.DataFrame(fold_rows).to_csv(output_dir / "ftt_v2_fold_metrics.csv", index=False)
    pd.DataFrame(post_rows).to_json(output_dir / "ftt_v2_postprocess_folds.json", orient="records", indent=2)
    np.savez_compressed(output_dir / "ftt_v2_probabilities.npz", oof=oof, test=test_proba, y=y, fold_id=fold_id, classes=classes, train_ids=train_ids, test_ids=test_ids, class_prior=prior)
    save_submission(sample, label_encoder, test_proba.argmax(axis=1), output_dir / "submission_ftt_v2_raw.csv")
    save_submission(sample, label_encoder, prior_prediction(test_proba, prior, beta=1.0), output_dir / "submission_ftt_v2_inverse_prior.csv")
    save_submission(sample, label_encoder, prior_prediction(test_proba, prior, beta=full_beta), output_dir / "submission_ftt_v2_prior_beta.csv")
    save_submission(sample, label_encoder, np.argmax(test_proba * full_multipliers, axis=1), output_dir / "submission_ftt_v2_multiplier.csv")
    summary = {"config": config, "classes": classes.tolist(), "class_prior": prior.tolist(), "feature_count": 52, "raw_feature_count": len(feature_columns), "exact_value_te_feature_count": len(feature_columns) * 3, "comparison": comparison.to_dict(orient="records"), "postprocess_crossfit_folds": post_rows, "full_oof_beta_for_test": full_beta, "full_oof_beta_score_optimistic": full_beta_score, "full_oof_multipliers_for_test": full_multipliers.tolist(), "full_oof_multiplier_score_optimistic": full_multiplier_score, "train_id_sha256": array_sha256(train_ids), "test_id_sha256": array_sha256(test_ids), "oof_sha256": array_sha256(oof), "test_proba_sha256": array_sha256(test_proba), "elapsed_seconds": time.time() - started}
    (output_dir / "ftt_v2_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(comparison.to_string(index=False), flush=True)
    print(json.dumps({"full_beta": full_beta, "full_multipliers": full_multipliers.tolist(), "elapsed_seconds": time.time() - started}), flush=True)


## 10. 실행

마지막으로 `main()`을 실행합니다.

전체 실행 결과로는 모델 probability, fold별 지표, decision-rule 비교표, 네 가지 submission, 실험 설정/요약 파일이 `output-dir`에 저장됩니다.


In [ ]:
if __name__ == "__main__":
    main()


---

## 최종 해석

이번 결과를 **“FT-Transformer라서 무조건 성능이 올랐다”**고 해석하지는 않습니다.

성능 향상에는 다음 요소들이 함께 작동했습니다.

- class-specific Target Encoding으로 타깃 관련 정보 보강
- FT-Transformer의 attention 기반 feature interaction 학습
- 7-fold × 4-member ensemble로 variance 감소
- Balanced Accuracy에 맞춘 class imbalance correction

그중 실제 Kaggle test에서는 **fixed inverse-prior correction**이 가장 안정적으로 일반화되어 현재 최고 점수인 **Private 0.95066**을 기록했습니다.
